In [11]:
with open("book_training_data.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
print('total number of characters: ', len(raw_text))
print(raw_text[:1199])


total number of characters:  456795
Virus programming (basics) #1...
This section is dedicated to those who would like to write a virus, but don't have the
knowledge to do so.  First of all, writing a virus is no big deal.  It is an easy project, but one
which requires some basic programmin g skills, and the desire to write a virus!  If either of
these is missing, writing a virus would be tedious indeed!.
 Well, if you meet these requisites, keep reading this article....
               JE   READ
               JNE  FUCK_YOU!
READ:
The survival of a virus is based in its ability to reproduce.  "So how the fuck do I make a
program reproduce?", you might ask. Simple, by getting it to copy itself to other files....
The functional logic of a virus is as follows:
1- Search for a file to infect
2- Open the file to see if it is infected
3- If infected, search for another file
4- Else, infect the file
5- Return control to the host program.
The following is an example of a simple virus:
;*******

In [16]:
import re
text = raw_text[:99]
result = re.split(r'([,.:;?_!"()\#]|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Virus', 'programming', '(', 'basics', ')', '#', '1', '.', '.', '.', 'This', 'section', 'is', 'dedicated', 'to', 'those', 'who', 'would', 'like', 'to', 'write', 'a', 'virus']


In [20]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['Virus', 'programming', '(', 'basics', ')', '#1', '.', '.', '.', 'This', 'section', 'is', 'dedicated', 'to', 'those', 'who', 'would', 'like', 'to', 'write', 'a', 'virus', ',', 'but', 'don', "'", 't', 'have', 'the', 'knowledge']


In [21]:
print(len(preprocessed))

88214


In [22]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

13095


In [23]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [28]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 5:
    break

('\x1a', 0)
('!', 1)
('"', 2)
('#', 3)
('##', 4)
('############', 5)


In [26]:
class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}
  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()#\']|--|\s)', text)

    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [29]:
tokenizer = SimpleTokenizerV1(vocab)

text = raw_text[200:300]
ids = tokenizer.encode(text)
print(ids)

[13002, 11269, 111, 7961, 10852, 12896, 11579, 12020, 7761, 11265, 9448, 11976, 111, 7557, 12395, 8654, 12467, 12985, 7344, 12796, 1]


In [32]:
tokenizer.decode(ids)

'y project, but one which requires some basic programmin g skills, and the desire to write a virus!'

In [33]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [34]:
len(vocab.items())

13097

In [35]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

('ŒLECTRONIC', 13092)
('ŒLectronic', 13093)
('Œzine', 13094)
('<|endoftext|>', 13095)
('<|unk|>', 13096)


In [37]:
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = { i:s for s, i in vocab.items()}
  def encode(self, text):
    preprocessed = re.split(r'([,.:;?!"()#\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [
        item if item in self.str_to_int
        else "<|unk|>" for item in preprocessed
    ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.:;?!"()#\'])', r'\1', text)
    return text

In [39]:
tokenizer = SimpleTokenizerV2(vocab)
text1 = "hello, I am a Virus!"
text2 = raw_text[100:200]
text = " <|endoftext|> ".join((text1, text2))

print(text)

hello, I am a Virus! <|endoftext|>  but don't have the
knowledge to do so.  First of all, writing a virus is no big deal.  It is an eas


In [40]:
tokenizer.encode(text)

[9669,
 111,
 4293,
 7535,
 7344,
 6814,
 1,
 13095,
 7961,
 8836,
 44,
 12290,
 9645,
 12395,
 10126,
 12467,
 8818,
 12005,
 244,
 3872,
 10814,
 7500,
 111,
 12989,
 7344,
 12796,
 10017,
 10731,
 7832,
 8560,
 244,
 4466,
 10017,
 7546,
 13096]

In [41]:
tokenizer.decode(tokenizer.encode(text))

"hello, I am a Virus! <|endoftext|> but don' t have the knowledge to do so. First of all, writing a virus is no big deal. It is an <|unk|>"